# AURORA-v3: Two-Mode TKG Forecasting

- **YAGO/WIKI** → `copy_only=True` → pure calibrated copy (83% H@1 on YAGO)
- **ICEWS18/GDELT** → `fixed_gate` → copy + neural (no oscillation)

**Dataset setup:** Upload your TKG data as a Kaggle dataset named `tkg-elite2`
with folder structure:
```
tkg-elite2/
  YAGO/train.txt  valid.txt  test.txt
  WIKI/train.txt  valid.txt  test.txt
  ICEWS18/...
  GDELT/...
```

In [ ]:
# ── 1. CONFIG ─────────────────────────────────────────────────────────────────
DATASET   = "YAGO"       # YAGO | WIKI | ICEWS18 | GDELT
DATA_DIR  = "/kaggle/input/tkg-elite2"
SAVE_DIR  = "/kaggle/working/checkpoints"
LOG_DIR   = "/kaggle/working/logs"
GPU_IDX   = 0            # always 0 on Kaggle
SEED      = 42

In [ ]:
# ── 2. IMPORTS ────────────────────────────────────────────────────────────────
import os, time, json, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.amp import GradScaler, autocast
from torch.nn.utils import clip_grad_norm_
from collections import defaultdict
from typing import Dict, List, Tuple
from tqdm.auto import tqdm

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device(f"cuda:{GPU_IDX}" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(GPU_IDX)}")
    print(f"Memory: {torch.cuda.get_device_properties(GPU_IDX).total_memory/1e9:.1f} GB")

In [ ]:
# ── 3. PER-DATASET CONFIG ─────────────────────────────────────────────────────
CONFIGS = {
    "YAGO": dict(
        embed_dim=256, k_neighbors=64, hist_len=20, gru_layers=2, dropout=0.2,
        copy_lambda=0.02, recency_steps=7, recency_boost=15.0, use_entity_copy=True,
        copy_only=True, fixed_gate=1.0,
        epochs=50, batch_size=512, lr=1e-3, warmup_ratio=0.1,
        weight_decay=1e-4, grad_clip=1.0,
        alpha_infonce=0.0, infonce_temp=0.07, label_smoothing=0.05,
        eval_every=1, hits_at=(1, 3, 10),
    ),
    "WIKI": dict(
        embed_dim=256, k_neighbors=64, hist_len=20, gru_layers=2, dropout=0.2,
        copy_lambda=0.05, recency_steps=5, recency_boost=10.0, use_entity_copy=True,
        copy_only=True, fixed_gate=1.0,
        epochs=50, batch_size=512, lr=1e-3, warmup_ratio=0.1,
        weight_decay=1e-4, grad_clip=1.0,
        alpha_infonce=0.0, infonce_temp=0.07, label_smoothing=0.05,
        eval_every=1, hits_at=(1, 3, 10),
    ),
    "ICEWS18": dict(
        embed_dim=256, k_neighbors=32, hist_len=20, gru_layers=2, dropout=0.3,
        copy_lambda=0.5, recency_steps=2, recency_boost=4.0, use_entity_copy=True,
        copy_only=False, fixed_gate=0.25,
        epochs=60, batch_size=1024, lr=2e-4, warmup_ratio=0.1,
        weight_decay=1e-4, grad_clip=1.0,
        alpha_infonce=0.4, infonce_temp=0.07, label_smoothing=0.1,
        eval_every=1, hits_at=(1, 3, 10),
    ),
    "GDELT": dict(
        embed_dim=256, k_neighbors=32, hist_len=15, gru_layers=2, dropout=0.3,
        copy_lambda=0.5, recency_steps=2, recency_boost=4.0, use_entity_copy=True,
        copy_only=False, fixed_gate=0.20,
        epochs=40, batch_size=512, lr=2e-4, warmup_ratio=0.1,
        weight_decay=1e-4, grad_clip=1.0,
        alpha_infonce=0.4, infonce_temp=0.07, label_smoothing=0.1,
        eval_every=1, hits_at=(1, 3, 10),
    ),
}

cfg = type("CFG", (), {"dataset": DATASET, "data_dir": DATA_DIR,
                        "seed": SEED, "use_inverse": True,
                        **CONFIGS[DATASET]})()
print(f"Dataset: {DATASET}  mode={'copy-only' if cfg.copy_only else f'fixed-gate={cfg.fixed_gate}'}")

In [ ]:
# ── 4. DATA LOADING ───────────────────────────────────────────────────────────
def load_quadruples(path):
    quads = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                quads.append([int(parts[0]), int(parts[1]),
                               int(parts[2]), int(parts[3])])
    return np.array(quads, dtype=np.int32)


def get_step(dataset):
    return 24 if dataset in ("ICEWS18", "ICEWS14") else 1


class GraphIndex:
    def __init__(self, quads_all, step=1):
        self.step = step
        self.all_answers = defaultdict(set)
        for s, r, o, t in quads_all:
            self.all_answers[(int(s), int(r), int(t))].add(int(o))
        self._by_time_sub = defaultdict(list)
        for s, r, o, t in quads_all:
            self._by_time_sub[(int(t), int(s))].append((int(r), int(o)))
        _sr_raw = defaultdict(list)
        for s, r, o, t in quads_all:
            _sr_raw[(int(s), int(r), int(o))].append(int(t))
        self._sr_objs = defaultdict(dict)
        for (s, r, o), times in _sr_raw.items():
            self._sr_objs[(s, r)][o] = sorted(times)
        self._by_sub = defaultdict(list)
        for s, r, o, t in quads_all:
            self._by_sub[int(s)].append((int(o), int(t), int(r)))

    def get_rel_copy_scores(self, sub, rel, query_time, num_entities,
                             copy_lambda, recency_steps, recency_boost):
        scores = np.zeros(num_entities, dtype=np.float32)
        step = max(self.step, 1)
        thr  = recency_steps * step
        for o, times in self._sr_objs.get((sub, rel), {}).items():
            ts = [tt for tt in times if tt < query_time]
            if not ts: continue
            last_t = max(ts)
            decay  = np.exp(-copy_lambda * (query_time - last_t) / step)
            score  = np.log1p(len(ts)) * decay
            if (query_time - last_t) <= thr:
                score *= recency_boost
            if o < num_entities:
                scores[o] = score
        return scores

    def get_ent_copy_scores(self, sub, query_time, num_entities,
                             copy_lambda, recency_steps, recency_boost):
        scores = np.zeros(num_entities, dtype=np.float32)
        step   = max(self.step, 1)
        thr    = recency_steps * step
        by_obj = defaultdict(list)
        for o, t, _ in self._by_sub.get(sub, []):
            if t < query_time:
                by_obj[o].append(t)
        for o, ts in by_obj.items():
            if o >= num_entities: continue
            last_t = max(ts)
            decay  = np.exp(-copy_lambda * (query_time - last_t) / step)
            score  = np.log1p(len(ts)) * decay
            if (query_time - last_t) <= thr:
                score *= recency_boost
            scores[o] = score
        return scores

    def get_neighbors(self, sub, query_time, hist_len, k_neighbors, rng):
        H, K = hist_len, k_neighbors
        ne = np.zeros((H, K), dtype=np.int32)
        nr = np.zeros((H, K), dtype=np.int32)
        nm = np.zeros((H, K), dtype=bool)
        t  = query_time - self.step
        for h in range(H):
            if t < 0: break
            facts = self._by_time_sub.get((t, sub), [])
            if facts:
                chosen = rng.sample(facts, min(len(facts), K))
                for k, (r, o) in enumerate(chosen):
                    ne[h, k] = o; nr[h, k] = r; nm[h, k] = True
            t -= self.step
        return ne, nr, nm


class CFDataset(Dataset):
    def __init__(self, quads, index, num_entities, cfg,
                 use_inverse=False, num_relations=None, desc=""):
        step = get_step(cfg.dataset)
        if use_inverse and num_relations is not None:
            inv  = np.stack([quads[:,2], quads[:,1]+num_relations,
                             quads[:,0], quads[:,3]], axis=1).astype(np.int32)
            data = np.concatenate([quads, inv], axis=0)
        else:
            data = quads.copy()
        self.data = data
        N = len(data)
        H, K = cfg.hist_len, cfg.k_neighbors
        rng  = random.Random(cfg.seed)

        print(f"  [{desc}] Pre-computing neighborhoods ({N:,}, H={H}, K={K})…", flush=True)
        ne = np.zeros((N, H, K), dtype=np.int16)
        nr = np.zeros((N, H, K), dtype=np.int16)
        nm = np.zeros((N, H, K), dtype=bool)
        for i in range(N):
            s, _, _, t = data[i]
            ne_i, nr_i, nm_i = index.get_neighbors(int(s), int(t), H, K, rng)
            ne[i] = ne_i.clip(-32768, 32767).astype(np.int16)
            nr[i] = nr_i.clip(-32768, 32767).astype(np.int16)
            nm[i] = nm_i
            if (i+1) % 200_000 == 0:
                print(f"    {i+1:,}/{N:,}", flush=True)
        self._ne, self._nr, self._nm = ne, nr, nm

        print(f"  [{desc}] Pre-computing copy scores…", flush=True)
        rc_idx, rc_val = [None]*N, [None]*N
        ec_idx, ec_val = [None]*N, [None]*N
        for i in range(N):
            s, r, _, t = data[i]
            rc = index.get_rel_copy_scores(int(s), int(r), int(t), num_entities,
                                           cfg.copy_lambda, cfg.recency_steps,
                                           cfg.recency_boost)
            nz = rc.nonzero()[0]
            rc_idx[i], rc_val[i] = nz.astype(np.int32), rc[nz]
            if cfg.use_entity_copy:
                ec = index.get_ent_copy_scores(int(s), int(t), num_entities,
                                               cfg.copy_lambda, cfg.recency_steps,
                                               cfg.recency_boost)
                nze = ec.nonzero()[0]
                ec_idx[i], ec_val[i] = nze.astype(np.int32), ec[nze]
            else:
                ec_idx[i] = np.zeros(0, dtype=np.int32)
                ec_val[i] = np.zeros(0, dtype=np.float32)
            if (i+1) % 200_000 == 0:
                print(f"    {i+1:,}/{N:,}", flush=True)
        self._rc_idx, self._rc_val = rc_idx, rc_val
        self._ec_idx, self._ec_val = ec_idx, ec_val
        self._num_ent = num_entities
        print(f"  [{desc}] Done.", flush=True)

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        s, r, o, t = self.data[idx]
        ne = torch.from_numpy(self._ne[idx].astype(np.int32)).long()
        nr = torch.from_numpy(self._nr[idx].astype(np.int32)).long()
        nm = torch.from_numpy(self._nm[idx])
        return (int(s), int(r), int(o), int(t),
                ne, nr, nm,
                self._rc_idx[idx], self._rc_val[idx],
                self._ec_idx[idx], self._ec_val[idx])


def cf_collate(batch):
    (subs, rels, objs, times,
     ne_l, nr_l, nm_l,
     rc_idx_l, rc_val_l, ec_idx_l, ec_val_l) = zip(*batch)
    subs  = torch.tensor(subs,  dtype=torch.long)
    rels  = torch.tensor(rels,  dtype=torch.long)
    objs  = torch.tensor(objs,  dtype=torch.long)
    times = torch.tensor(times, dtype=torch.long)
    ne = torch.stack(ne_l); nr = torch.stack(nr_l); nm = torch.stack(nm_l)
    all_idx = np.concatenate(
        [x for x in rc_idx_l if len(x) > 0] +
        [x for x in ec_idx_l if len(x) > 0])
    N = int(all_idx.max()) + 1 if len(all_idx) > 0 else 1
    B = len(subs)
    rc = torch.zeros(B, N, dtype=torch.float32)
    ec = torch.zeros(B, N, dtype=torch.float32)
    for i in range(B):
        if len(rc_idx_l[i]) > 0:
            rc[i, rc_idx_l[i]] = torch.from_numpy(rc_val_l[i])
        if len(ec_idx_l[i]) > 0:
            ec[i, ec_idx_l[i]] = torch.from_numpy(ec_val_l[i])
    return subs, rels, objs, times, ne, nr, nm, rc, ec


class CFDataLoader:
    def __init__(self, cfg):
        self.cfg  = cfg
        base      = os.path.join(cfg.data_dir, cfg.dataset)
        self.step = get_step(cfg.dataset)
        train_q = load_quadruples(os.path.join(base, "train.txt"))
        valid_q = load_quadruples(os.path.join(base, "valid.txt"))
        test_q  = load_quadruples(os.path.join(base, "test.txt"))
        all_q = np.concatenate([train_q, valid_q, test_q], axis=0)
        self.num_entities  = int(all_q[:, [0,2]].max()) + 1
        self.num_relations = int(all_q[:, 1].max()) + 1
        print(f"[{cfg.dataset}] entities={self.num_entities:,}  "
              f"relations={self.num_relations}  "
              f"train={len(train_q):,}  valid={len(valid_q):,}  "
              f"test={len(test_q):,}  step={self.step}")
        self.index = GraphIndex(all_q, step=self.step)
        self.train_set = CFDataset(train_q, self.index, self.num_entities, cfg,
                                   use_inverse=cfg.use_inverse,
                                   num_relations=self.num_relations, desc="train")
        self.valid_set = CFDataset(valid_q, self.index, self.num_entities,
                                   cfg, desc="valid")
        self.test_set  = CFDataset(test_q,  self.index, self.num_entities,
                                   cfg, desc="test")
        print("Datasets ready.")

print("Data code ready.")

In [ ]:
# ── 5. MODEL ──────────────────────────────────────────────────────────────────
class AURORAv3Model(nn.Module):
    def __init__(self, num_entities, num_relations, cfg):
        super().__init__()
        d = cfg.embed_dim
        R = num_relations * 2
        self.num_entities = num_entities
        self.copy_only  = cfg.copy_only
        self.fixed_gate = cfg.fixed_gate

        self.rel_blend   = nn.Embedding(R, 1)
        self.rel_temp    = nn.Embedding(R, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))
        nn.init.zeros_(self.rel_blend.weight)
        nn.init.ones_(self.rel_temp.weight)

        if not self.copy_only:
            self.ent_emb = nn.Embedding(num_entities, d)
            self.rel_emb = nn.Embedding(R, d)
            nn.init.xavier_uniform_(self.ent_emb.weight)
            nn.init.xavier_uniform_(self.rel_emb.weight)
            self.snap_norm = nn.LayerNorm(d)
            self.gru = nn.GRU(d, d, num_layers=cfg.gru_layers,
                               batch_first=True,
                               dropout=cfg.dropout if cfg.gru_layers > 1 else 0.0)
            self.query_proj = nn.Sequential(
                nn.Linear(d*3, d*2), nn.LayerNorm(d*2), nn.GELU(),
                nn.Dropout(cfg.dropout), nn.Linear(d*2, d), nn.LayerNorm(d))

    def _pad(self, x):
        N = self.num_entities
        if x.shape[1] < N: return F.pad(x, (0, N - x.shape[1]))
        return x[:, :N]

    def _copy_logit(self, rels, rel_copy, ent_copy):
        rel_copy = self._pad(rel_copy)
        ent_copy = self._pad(ent_copy)
        w    = torch.sigmoid(self.rel_blend(rels))
        cs   = w * rel_copy + (1 - w) * ent_copy
        temp = F.softplus(self.rel_temp(rels))
        return torch.log1p(cs) * temp + self.global_bias, cs

    def _encode_history(self, subs, ne, nm):
        B, H, K = ne.shape
        ne_c = ne.clamp(0, self.num_entities - 1)
        h_neigh = self.ent_emb(ne_c)
        mask_f  = nm.float().unsqueeze(-1)
        counts  = mask_f.sum(2).clamp(min=1.0)
        h_snap  = (h_neigh * mask_f).sum(2) / counts
        h_snap  = self.snap_norm(h_snap)
        h_s_exp = self.ent_emb(subs).unsqueeze(1).expand_as(h_snap)
        no_nb   = (counts.squeeze(-1) == 0).float().unsqueeze(-1)
        h_snap  = h_snap * (1 - no_nb) + h_s_exp * no_nb
        _, h_last = self.gru(h_snap)
        return h_last[-1]

    def forward(self, subs, rels, ne, nr, nm, rel_copy, ent_copy):
        copy_logit, _ = self._copy_logit(rels, rel_copy, ent_copy)
        if self.copy_only:
            return copy_logit, None
        h_hist = self._encode_history(subs, ne, nm)
        query  = self.query_proj(
            torch.cat([self.ent_emb(subs), self.rel_emb(rels), h_hist], -1))
        neural_logit = query @ self.ent_emb.weight.T
        g = self.fixed_gate
        return g * copy_logit + (1 - g) * neural_logit, query

    def get_query(self, subs, rels, ne, nm):
        if self.copy_only: return None
        h_hist = self._encode_history(subs, ne, nm)
        return self.query_proj(
            torch.cat([self.ent_emb(subs), self.rel_emb(rels), h_hist], -1))

print("Model code ready.")

In [ ]:
# ── 6. LOSS ───────────────────────────────────────────────────────────────────
def label_smooth_ce(logits, targets, smoothing=0.05):
    log_prob = F.log_softmax(logits, dim=-1)
    nll    = -log_prob.gather(1, targets.unsqueeze(1)).squeeze(1)
    smooth = -log_prob.mean(dim=-1)
    return ((1 - smoothing) * nll + smoothing * smooth).mean()

def infonce_loss(query, ent_emb, targets, temperature=0.07):
    q = F.normalize(query, dim=-1)
    e = F.normalize(ent_emb[targets], dim=-1)
    sim = q @ e.T / temperature
    labels = torch.arange(sim.size(0), device=sim.device)
    return F.cross_entropy(sim, labels)

print("Loss code ready.")

In [ ]:
# ── 7. EVALUATION ─────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, split="valid", hits_at=(1, 3, 10)):
    model.eval()
    dataset = loader.valid_set if split == "valid" else loader.test_set
    dl = DataLoader(dataset, batch_size=256, shuffle=False,
                    num_workers=0, collate_fn=cf_collate)
    mrr_sum = 0.0
    hits = {k: 0.0 for k in hits_at}
    total = 0
    N = loader.num_entities
    for batch in tqdm(dl, desc=f"Eval [{split}]", leave=False):
        subs, rels, objs, times, ne, nr, nm, rel_copy, ent_copy = batch
        subs = subs.to(device); rels = rels.to(device); objs = objs.to(device)
        ne   = ne.to(device);   nm   = nm.to(device)
        if rel_copy.shape[1] < N:
            rel_copy = torch.cat([rel_copy,
                torch.zeros(rel_copy.shape[0], N-rel_copy.shape[1])], 1)
        if ent_copy.shape[1] < N:
            ent_copy = torch.cat([ent_copy,
                torch.zeros(ent_copy.shape[0], N-ent_copy.shape[1])], 1)
        rel_copy = rel_copy.to(device); ent_copy = ent_copy.to(device)
        logits, _ = model(subs, rels, ne, nr, nm, rel_copy, ent_copy)
        for i in range(subs.size(0)):
            s, r, t = subs[i].item(), rels[i].item(), times[i].item()
            o_true  = objs[i].item()
            sc = logits[i].clone()
            for o_f in loader.index.all_answers.get((s, r, t), set()):
                if o_f != o_true: sc[o_f] = float("-inf")
            rank = (sc > sc[o_true]).sum().item() + 1
            mrr_sum += 1.0 / rank
            for k in hits_at: hits[k] += float(rank <= k)
            total += 1
    res = {"MRR": mrr_sum/total}
    res.update({f"Hits@{k}": hits[k]/total for k in hits_at})
    return res

print("Evaluation code ready.")

In [ ]:
# ── 8. LOAD DATA ──────────────────────────────────────────────────────────────
loader = CFDataLoader(cfg)

In [ ]:
# ── 9. BUILD MODEL ────────────────────────────────────────────────────────────
model = AURORAv3Model(
    num_entities=loader.num_entities,
    num_relations=loader.num_relations,
    cfg=cfg,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
mode = "copy-only" if cfg.copy_only else f"fixed-gate={cfg.fixed_gate}"
print(f"[Model] AURORA-v3  params={n_params:,}  mode={mode}")

use_amp = (device.type == "cuda")
scaler  = GradScaler("cuda", enabled=use_amp)

optim = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
warmup_ep = max(1, int(cfg.epochs * cfg.warmup_ratio))
warmup    = LinearLR(optim, start_factor=0.1, end_factor=1.0,
                     total_iters=warmup_ep)
cosine    = CosineAnnealingLR(optim, T_max=cfg.epochs - warmup_ep,
                               eta_min=cfg.lr * 0.01)
sched     = SequentialLR(optim, [warmup, cosine], milestones=[warmup_ep])

In [ ]:
# ── 10. TRAINING LOOP ─────────────────────────────────────────────────────────
HEADER = (f"{'Ep':>4} | {'Time':>6} | {'Loss':>8} | "
          f"{'MRR':>7} {'H@1':>7} {'H@3':>7} {'H@10':>7} | {'LR':>9}")
SEP = "-" * len(HEADER)
print(f"\n{'='*len(HEADER)}")
print(f"  AURORA-v3 | {DATASET} | {mode}")
print(f"{'='*len(HEADER)}\n")
print(HEADER); print(SEP)

best_mrr, best_ep = 0.0, 0
log_path = os.path.join(LOG_DIR, f"{DATASET}_v3_log.jsonl")
N = loader.num_entities

for ep in range(1, cfg.epochs + 1):
    model.train()
    t0 = time.time()

    # Kaggle: num_workers=2, persistent_workers=False
    dl = DataLoader(
        loader.train_set, batch_size=cfg.batch_size,
        shuffle=True, num_workers=2,
        pin_memory=(device.type == "cuda"),
        drop_last=True, collate_fn=cf_collate,
        persistent_workers=False,
    )

    tot_loss = 0.0; n = 0
    for batch in tqdm(dl, desc=f" ep{ep}", leave=False):
        subs, rels, objs, times, ne, nr, nm, rel_copy, ent_copy = batch
        subs = subs.to(device); rels = rels.to(device); objs = objs.to(device)
        ne   = ne.to(device);   nm   = nm.to(device)
        if rel_copy.shape[1] < N:
            rel_copy = torch.cat([rel_copy,
                torch.zeros(rel_copy.shape[0], N-rel_copy.shape[1])], 1)
        if ent_copy.shape[1] < N:
            ent_copy = torch.cat([ent_copy,
                torch.zeros(ent_copy.shape[0], N-ent_copy.shape[1])], 1)
        rel_copy = rel_copy.to(device); ent_copy = ent_copy.to(device)

        optim.zero_grad()
        with autocast("cuda", dtype=torch.bfloat16, enabled=use_amp):
            logits, query = model(subs, rels, ne, nr, nm, rel_copy, ent_copy)
            loss = label_smooth_ce(logits, objs, cfg.label_smoothing)
            if query is not None and cfg.alpha_infonce > 0:
                nce  = infonce_loss(query, model.ent_emb.weight, objs,
                                    cfg.infonce_temp)
                loss = (1 - cfg.alpha_infonce)*loss + cfg.alpha_infonce*nce

        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optim); scaler.update()
        tot_loss += loss.item(); n += 1

    sched.step()
    elapsed = time.time() - t0
    lr_now  = sched.get_last_lr()[0]

    metrics = {}
    if ep % cfg.eval_every == 0:
        metrics = evaluate(model, loader, split="valid",
                           hits_at=cfg.hits_at)

    is_best = metrics.get("MRR", 0) > best_mrr
    if is_best:
        best_mrr = metrics["MRR"]; best_ep = ep
        torch.save({"model": model.state_dict()},
                   os.path.join(SAVE_DIR, f"{DATASET}_v3_best.pt"))

    star = "★" if is_best else " "
    row = (f"{star}{ep:>3} | {elapsed:>5.1f}s | {tot_loss/n:>8.4f} | "
           f"{metrics.get('MRR',0):>7.4f} {metrics.get('Hits@1',0):>7.4f} "
           f"{metrics.get('Hits@3',0):>7.4f} {metrics.get('Hits@10',0):>7.4f} | "
           f"{lr_now:>9.2e}")
    print(row)

    with open(log_path, "a") as f:
        f.write(json.dumps({"epoch": ep, "loss": tot_loss/n,
                            "metrics": metrics, "lr": lr_now}) + "\n")

print(SEP)
print(f"\n★  Best valid MRR = {best_mrr:.4f}  (epoch {best_ep})")

In [ ]:
# ── 11. FINAL TEST ────────────────────────────────────────────────────────────
ck_path = os.path.join(SAVE_DIR, f"{DATASET}_v3_best.pt")
if os.path.exists(ck_path):
    ck = torch.load(ck_path, map_location=device, weights_only=True)
    model.load_state_dict(ck["model"])
    print("Best checkpoint loaded.")

test_m = evaluate(model, loader, split="test", hits_at=cfg.hits_at)

print(f"\n{'='*50}")
print(f"  TEST RESULTS — {DATASET}  [{mode}]")
print(f"{'='*50}")
for k, v in test_m.items():
    print(f"  {k:<12}: {v:.4f}")
print(f"{'='*50}")

out = os.path.join(SAVE_DIR, f"{DATASET}_v3_results.json")
with open(out, "w") as f:
    json.dump({"dataset": DATASET, "mode": mode,
               "best_epoch": best_ep, "valid_mrr": best_mrr,
               "test": test_m}, f, indent=2)
print(f"Saved → {out}")